# CALM-VAD — pose extraction + evaluation (Colab)

**Only step that needs a GPU: #4 (pose extraction).** The dataset is downloaded
into Colab's own ~100 GB scratch disk — **not** your Google Drive — so a 40 GB
benchmark never touches your 15 GB Drive. Only the small pose JSON + report are
saved to Drive at the end.

Flow: env → code → **get dataset into /content** → extract poses (GPU) →
harness (CPU) → download poses+report.

**First:** `Runtime → Change runtime type → T4 GPU`.

## 1 · Environment

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics scikit-learn scipy pyyaml opencv-python-headless gdown
import torch; print('CUDA available:', torch.cuda.is_available())

## 2 · Get the code

Set `REPO_URL` to your pushed GitHub repo, **or** leave it `''` and the cell
asks you to upload a zip of the `sentrix` project.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/FaizanAbbas512/Sentrix.git'
import os, sys
if REPO_URL:
    !rm -rf /content/sentrix && git clone --depth 1 $REPO_URL /content/sentrix
else:
    from google.colab import files
    print('Upload a zip of the sentrix project (must contain the calm/ folder):')
    up = files.upload(); zname = next(iter(up))
    !rm -rf /content/sentrix && mkdir -p /content/sentrix && unzip -q "$zname" -d /content/sentrix
    subs = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in subs and len(subs) == 1:
        !cp -r /content/sentrix/{subs[0]}/* /content/sentrix/
os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm'), 'calm/ not found - check the repo/zip layout'
!mkdir -p data/pose results /content/data/videos /content/data/gt /content/raw
print('cwd:', os.getcwd(), '| calm:', sorted(os.listdir('calm')))

In [ ]:
# sanity: the maths works before we touch any data (~2 s)
!python -m calm.selftest

## 3 · Get the dataset **into Colab's disk** (not Drive)

Set **one** of `GDRIVE_ID` / `DIRECT_URL` / `KAGGLE_DS` below, then run.
The cell downloads, extracts, and **auto-finds** the videos and the ground
truth anywhere in the archive (`.npy` masks, `.mat` pixel masks — converted
for you — `.txt`, or `.json`).

You only need the **testing** split — CALM-VAD trains nothing.

Where to get a link (verify on the dataset's own page — academic links rot):
* **CUHK Avenue** (~2 GB, whole set): `DIRECT_URL = 'http://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/Avenue_Dataset.zip'`
* **UCSD Ped2** (small, low-res — pose is weak): `DIRECT_URL = 'http://www.svcl.ucsd.edu/projects/anomaly/UCSD_Anomaly_Dataset.tar.gz'`
* **ShanghaiTech**: get a Drive share link from https://svip-lab.github.io/dataset/campus_dataset.html → put the id/link in `GDRIVE_ID`
* **UBnormal**: https://github.com/lilygeorgescu/UBnormal  |  **NWPU**: https://campusvad.github.io/  |  **CHAD**: https://github.com/TeCSAR-UNCC/CHAD
* **Kaggle mirrors**: search kaggle.com/datasets for the name → `KAGGLE_DS = 'owner/slug'`

In [ ]:
import os, glob, shutil, zipfile, tarfile, numpy as np

GDRIVE_ID  = ''    # a Google-Drive file id OR full share link (uses gdown, fuzzy)
DIRECT_URL = ''    # a direct http(s) link to a .zip / .tar / .tar.gz
KAGGLE_DS  = ''    # 'owner/dataset-slug'  (will ask you to upload kaggle.json)

for d in ('/content/data/videos', '/content/data/gt', '/content/raw'):
    os.makedirs(d, exist_ok=True)
arc = None
if GDRIVE_ID:
    import gdown
    src = GDRIVE_ID if 'http' in GDRIVE_ID else f'https://drive.google.com/uc?id={GDRIVE_ID}'
    gdown.download(src, '/content/raw/ds', quiet=False, fuzzy=True); arc = '/content/raw/ds'
elif DIRECT_URL:
    !wget -q --show-progress -O /content/raw/ds "$DIRECT_URL"
    arc = '/content/raw/ds'
elif KAGGLE_DS:
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files as _f
        print('Upload kaggle.json  (kaggle.com > Settings > Create New Token):'); _f.upload()
        !mkdir -p /root/.kaggle && cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !pip -q install kaggle && kaggle datasets download -d "$KAGGLE_DS" -p /content/raw --unzip
else:
    raise SystemExit('Set GDRIVE_ID or DIRECT_URL or KAGGLE_DS above, then re-run.')

def _extract(path, dest):
    if zipfile.is_zipfile(path): zipfile.ZipFile(path).extractall(dest)
    elif tarfile.is_tarfile(path): tarfile.open(path).extractall(dest)
if arc and os.path.isfile(arc):
    print('extracting ...'); _extract(arc, '/content/raw')

# ---- auto-find videos (prefer a path containing 'test') ----
VEXT = ('.avi', '.mp4', '.mov', '.mkv')
vids = [p for p in glob.glob('/content/raw/**/*', recursive=True) if p.lower().endswith(VEXT)]
vids = [p for p in vids if 'train' not in p.lower()] or vids
for p in vids: shutil.copy(p, f"/content/data/videos/{os.path.basename(p)}")

# ---- auto-find ground truth ----
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
npy = [p for p in glob.glob('/content/raw/**/*.npy', recursive=True) if ('test' in p.lower() or 'mask' in p.lower())]
mat = [p for p in glob.glob('/content/raw/**/*.mat', recursive=True) if any(k in p.lower() for k in ('test','label','gt','mask'))]
txt = [p for p in glob.glob('/content/raw/**/*.txt', recursive=True) if any(k in p.lower() for k in ('test','label','gt'))]
if npy:
    for p in npy: shutil.copy(p, f'/content/data/gt/{stem(p)}.npy')
elif mat:
    from scipy.io import loadmat
    for p in mat:
        m = loadmat(p)
        arrs = [v for k, v in m.items() if not k.startswith('__') and getattr(v, 'ndim', 0)]
        if not arrs: continue
        a = np.asarray(max(arrs, key=lambda x: x.size))
        frame = ((a.reshape(-1, a.shape[-1]).sum(0) > 0).astype(int) if a.ndim >= 3
                 else (a.ravel() > 0).astype(int))
        np.save(f'/content/data/gt/{stem(p)}.npy', frame)
elif txt:
    for p in txt: shutil.copy(p, f'/content/data/gt/{stem(p)}.txt')

nv = len(glob.glob('/content/data/videos/*')); ng = len(glob.glob('/content/data/gt/*'))
print(f'\nvideos: {nv}   ground-truth files: {ng}')
print('sample:', [os.path.basename(x) for x in glob.glob("/content/data/videos/*")[:4]])
if ng == 0:
    print('\nNOTE: no GT auto-found. Look in /content/raw for the label files and',
          'copy them into /content/data/gt yourself (as .npy / .txt / .json).')

## 4 · Extract poses — **GPU** (~15–25 min for a mid-size benchmark)

`--stride 2 --imgsz 512` roughly halves the time at a small accuracy cost.
Extract everything as `test`; the harness carves its own calibration split.

In [ ]:
TAG = 'avenue'   # name this run
!python -m calm.extract_poses \
  --videos /content/data/videos --gt /content/data/gt \
  --out data/pose/$TAG.json --split test \
  --weights yolo11n-pose.pt --imgsz 640 --device 0 --stride 1

## 5 · Evaluate — CPU-light, the 5-axis report

In [ ]:
!python -m calm.harness --generic data/pose/$TAG.json --tag $TAG
import json; r = json.load(open(f'results/calm_report_{TAG}.json'))
print('\n=== key numbers ===')
for s in r['streams']:
    print(f"  {s['label']:<26} AUC {s['frame_auc']:.3f}  eventF1 {s['event']['f1_avg']:.3f}",
          {k: v for k, v in s.items() if k.startswith('faph@')})
print('  calibration ECE raw->cal:', round(r['calibration']['ece_raw'],4), '->', round(r['calibration']['ece_cal'],4))

## 6 · Download poses + report (small — safe for Drive)

In [ ]:
!zip -qr /content/calm_$TAG.zip data/pose results
from google.colab import files; files.download(f'/content/calm_{TAG}.zip')
try:
    from google.colab import drive; drive.mount('/content/drive')
    !cp /content/calm_$TAG.zip /content/drive/MyDrive/ && echo 'also copied to Drive/MyDrive'
except Exception as e:
    print('Drive copy skipped:', e)

## 7 · More benchmarks / cross-dataset

Re-run cells 3–6 with a new link and a new `TAG` for each benchmark
(`nwpu`, `ubnormal`, `shanghaitech`, `chad`).

**Cross-dataset drop:** merge two pose files, one as `calib`, one as `test`:

In [ ]:
import json
def merge(fit_json, test_json, out_json):
    a = json.load(open(fit_json)); b = json.load(open(test_json))
    for c in a['clips']: c['split'] = 'calib'
    for c in b['clips']: c['split'] = 'test'
    json.dump({'fps': a['fps'], 'clips': a['clips'] + b['clips']}, open(out_json, 'w'))
    print('wrote', out_json)
# merge('data/pose/shanghaitech.json', 'data/pose/nwpu.json', 'data/pose/sht2nwpu.json')
# !python -m calm.harness --generic data/pose/sht2nwpu.json --tag sht2nwpu

## 8 · (Fastest) Option Z — use **pre-extracted skeletons**, skip the GPU

Repos like **STG-NF** and **MoCoDAD** publish ready skeletons for
HR-ShanghaiTech / HR-Avenue (+ frame masks). If you download those, you skip
the video download **and** pose extraction — just convert to the schema and
run the harness. Adapt the loader below to whatever layout the release uses.

In [ ]:
import glob, json, os, numpy as np
SKEL_DIR = '/content/skel'          # where you unpacked the released skeletons
OUT = 'data/pose/hr_shanghaitech.json'; FPS = 24

clips = []
for jp in sorted(glob.glob(f'{SKEL_DIR}/*.json')):     # <-- change if it's .npy
    per_frame = json.load(open(jp))                    # {frame_idx: [ {keypoints, idx}, ... ]}
    st = os.path.splitext(os.path.basename(jp))[0]
    idxs = sorted(int(k) for k in per_frame); n = (max(idxs) + 1) if idxs else 0
    frames = [[] for _ in range(n)]
    for k, ppl in per_frame.items():
        for pp in ppl:
            kp = np.asarray(pp['keypoints'], float).reshape(-1, 3)[:17].round(2).tolist()
            frames[int(k)].append({'id': int(pp.get('idx', -1)), 'keypoints': kp,
                                   'bbox': None, 'is_person': True, 'label': 'person'})
    gt = []; gp = f'{SKEL_DIR}/{st}_gt.npy'
    if os.path.exists(gp):
        mask = np.load(gp).astype(int).ravel(); i = 0
        while i < len(mask):
            if mask[i]:
                j = i
                while j < len(mask) and mask[j]: j += 1
                gt.append([i, j - 1]); i = j
            else: i += 1
    clips.append({'name': st, 'n_frames': n, 'split': 'test', 'gt': gt, 'frames': frames})

os.makedirs('data/pose', exist_ok=True)
json.dump({'fps': FPS, 'clips': clips}, open(OUT, 'w'))
print(f'wrote {OUT} ({len(clips)} clips) -> !python -m calm.harness --generic {OUT} --tag hr_shanghaitech')